In [203]:
import warnings
warnings.filterwarnings('ignore')

In [204]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [205]:
res = pd.read_csv("data/1976-2024-house.tab")
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
0,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,BILL DAVENPORT,DEMOCRAT,False,TOTAL,58906,157170,False,20250910,False
1,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,JACK EDWARDS,REPUBLICAN,False,TOTAL,98257,157170,False,20250910,False
2,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,WRITEIN,NaN,True,TOTAL,7,157170,False,20250910,False
3,1976,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,J CAROLE KEAHEY,DEMOCRAT,False,TOTAL,66288,156362,False,20250910,False
4,1976,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,"WILLIAM L \""BILL\"" DICKINSON",REPUBLICAN,False,TOTAL,90069,156362,False,20250910,False


In [206]:
res.columns.values

array(['year', 'state', 'state_po', 'state_fips', 'state_cen', 'state_ic',
       'office', 'district', 'stage', 'runoff', 'special', 'candidate',
       'party', 'writein', 'mode', 'candidatevotes', 'totalvotes',
       'unofficial', 'version', 'fusion_ticket'], dtype=object)

In [207]:
res = res[(res['year'] >= 2018) & (res['stage'] == 'GEN') & (res['mode'] == 'TOTAL')]
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
28277,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,BRADLEY BYRNE,REPUBLICAN,False,TOTAL,153228,242617,False,20250910,False
28278,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,ROBERT KENNEDY JR,DEMOCRAT,False,TOTAL,89226,242617,False,20250910,False
28279,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,WRITEIN,NaN,True,TOTAL,163,242617,False,20250910,False
28280,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,MARTHA ROBY,REPUBLICAN,False,TOTAL,138879,226230,False,20250910,False
28281,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,TABITHA ISNER,DEMOCRAT,False,TOTAL,86931,226230,False,20250910,False


In [208]:
res['party'].value_counts()

party
DEMOCRAT                   1711
REPUBLICAN                 1678
LIBERTARIAN                 389
INDEPENDENT                 184
CONSERVATIVE                 88
                           ... 
WRITE-IN (UNAFFILIATED)       1
POPULIST PARTY                1
ALOHA AINA PARTY              1
AMERICAN SHOPPING PARTY       1
REPUBLICAN, LIBERTARIAN       1
Name: count, Length: 131, dtype: int64

In [209]:
# Focus on two party vote share in the model - no need to wrangle with third parties for now
res = res[res['party'].isin(['DEMOCRAT', 'REPUBLICAN'])]
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
28277,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,BRADLEY BYRNE,REPUBLICAN,False,TOTAL,153228,242617,False,20250910,False
28278,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,ROBERT KENNEDY JR,DEMOCRAT,False,TOTAL,89226,242617,False,20250910,False
28280,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,MARTHA ROBY,REPUBLICAN,False,TOTAL,138879,226230,False,20250910,False
28281,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,TABITHA ISNER,DEMOCRAT,False,TOTAL,86931,226230,False,20250910,False
28283,2018,ALABAMA,AL,1,63,41,US HOUSE,3,GEN,NaN,False,MALLORY HAGAN,DEMOCRAT,False,TOTAL,83996,231915,False,20250910,False


In [210]:
res.shape

(3389, 20)

In [211]:
# aggfunc is sum to account for races where two or more candidates from the same party are running in the general,
# we sum the votes of these candidates together as part of calculating two-party vote share
data = pd.pivot_table(data=res, values='candidatevotes', columns=['party'], index=['year', 'state', 'state_po', 'special', 'district'], aggfunc='sum').reset_index()
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN
0,2018,ALABAMA,AL,False,1,89226.0,153228.0
1,2018,ALABAMA,AL,False,2,86931.0,138879.0
2,2018,ALABAMA,AL,False,3,83996.0,147770.0
3,2018,ALABAMA,AL,False,4,46492.0,184255.0
4,2018,ALABAMA,AL,False,5,101388.0,159063.0


In [212]:
data.duplicated(subset=['year', 'state', 'special', 'district']).any() # np.False_ --> no duplicates

np.False_

In [213]:
# Get candidates

def get_cands(year, state, special, district, party):
    df = res[
        (res['year'] == year) &
        (res['state'] == state) &
        (res['special'] == special) &
        (res['district'] == district) &
        (res['party'] == party)
    ]
    if df.shape[0] == 1:
        return df['candidate'].values[0]
    else:
        return repr(list(df['candidate'].values))

def get_totvotes(year, state, special, district):
    df = res[
        (res['year'] == year) &
        (res['state'] == state) &
        (res['special'] == special) &
        (res['district'] == district)
    ]
    return df['totalvotes'].values[0]

In [214]:
get_cands(2018, 'ALABAMA', False, 1, 'DEMOCRAT') # conventional

'ROBERT KENNEDY JR'

In [215]:
get_cands(2024, 'CALIFORNIA', False, 12, 'DEMOCRAT') # two people same party

"['JENNIFER TRAN', 'LATEEFAH SIMON']"

In [216]:
get_cands(2024, 'Massachusetts'.upper(), False, 4, 'REPUBLICAN') # running unopposed

'[]'

In [217]:
def get_dem_cand(year, state, special, district):
    return get_cands(year, state, special, district, 'DEMOCRAT')

def get_rep_cand(year, state, special, district):
    return get_cands(year, state, special, district, 'REPUBLICAN')

In [218]:
data['dem_cand'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_dem_cand(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data['rep_cand'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_rep_cand(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,dem_cand,rep_cand
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,ROBERT KENNEDY JR,BRADLEY BYRNE
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,TABITHA ISNER,MARTHA ROBY
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,MALLORY HAGAN,MIKE ROGERS
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,LEE AUMAN,ROBERT ADERHOLT
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,PETER JOFFRION,MO BROOKS


In [219]:
data = data.rename({'DEMOCRAT': 'dem', 'REPUBLICAN': 'rep'}, axis=1)
data.head()

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,ROBERT KENNEDY JR,BRADLEY BYRNE
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,TABITHA ISNER,MARTHA ROBY
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,MALLORY HAGAN,MIKE ROGERS
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,LEE AUMAN,ROBERT ADERHOLT
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,PETER JOFFRION,MO BROOKS


In [220]:
data.shape

(1742, 9)

In [221]:
data['totalvotes'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_totvotes(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,ROBERT KENNEDY JR,BRADLEY BYRNE,242617
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,TABITHA ISNER,MARTHA ROBY,226230
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,MALLORY HAGAN,MIKE ROGERS,231915
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,LEE AUMAN,ROBERT ADERHOLT,230969
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,PETER JOFFRION,MO BROOKS,260673


In [222]:
data.shape

(1742, 10)

In [223]:
data['2party_votes'] = data['dem'] + data['rep']
data.head(2)

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,ROBERT KENNEDY JR,BRADLEY BYRNE,242617,242454.0
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,TABITHA ISNER,MARTHA ROBY,226230,225810.0


In [224]:
data['dem_cand'] = data['dem_cand'].str.title()
data['rep_cand'] = data['rep_cand'].str.title()

In [225]:
data.head(2)

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,Robert Kennedy Jr,Bradley Byrne,242617,242454.0
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,Tabitha Isner,Martha Roby,226230,225810.0


In [226]:
data.shape

(1742, 11)

In [227]:
data['state'] = data['state'].str.title()
data.head(2)

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes
0,2018,Alabama,AL,False,1,89226.0,153228.0,Robert Kennedy Jr,Bradley Byrne,242617,242454.0
1,2018,Alabama,AL,False,2,86931.0,138879.0,Tabitha Isner,Martha Roby,226230,225810.0


In [228]:
# From: https://github.com/markjrieke/2022-midterm-forecasts/tree/main
rieke = pd.read_csv('data/rieke_historical_results_2018_2020.csv')
rieke.head()

,cycle,race,state,seat,candidate_name_DEM,candidate_name_REP,dem_votes,rep_votes,dem_incumbent,rep_incumbent,prev_seat
0,2018,House,Alabama,District 1,Robert Kennedy Jr.,Bradley Byrne,89226,153228,n,y,rep
1,2018,House,Alabama,District 2,Tabitha Isner,Martha Roby,86931,138879,n,y,rep
2,2018,House,Alabama,District 3,Mallory Hagan,Mike Rogers,83996,147770,n,y,rep
3,2018,House,Alabama,District 4,Lee Auman,Robert Aderholt,46492,184255,n,y,rep
4,2018,House,Alabama,District 5,Peter Joffrion,Mo Brooks,101388,159063,n,y,rep


In [229]:
rieke_22 = pd.read_csv('data/rieke_historical_results_2022.csv', encoding='latin-1')
rieke_22.head()

,cycle,race,state,seat,candidate_name_DEM,candidate_name_REP,dem_incumbent,rep_incumbent,pending
0,2022,House,Alabama,District 1,Remrey,Carl,n,y,NaN
1,2022,House,Alabama,District 2,Harvey-Hall,Moore,n,y,NaN
2,2022,House,Alabama,District 3,Veasey,Rogers,n,y,NaN
3,2022,House,Alabama,District 4,Neighbors,Aderholt,n,y,NaN
4,2022,House,Alabama,District 5,Warner-Stanton,Strong,n,n,NaN


In [230]:
rieke = pd.concat([rieke, rieke_22], axis=0)
rieke.head()

,cycle,race,state,seat,candidate_name_DEM,candidate_name_REP,dem_votes,rep_votes,dem_incumbent,rep_incumbent,prev_seat,pending
0,2018,House,Alabama,District 1,Robert Kennedy Jr.,Bradley Byrne,89226.0,153228.0,n,y,rep,NaN
1,2018,House,Alabama,District 2,Tabitha Isner,Martha Roby,86931.0,138879.0,n,y,rep,NaN
2,2018,House,Alabama,District 3,Mallory Hagan,Mike Rogers,83996.0,147770.0,n,y,rep,NaN
3,2018,House,Alabama,District 4,Lee Auman,Robert Aderholt,46492.0,184255.0,n,y,rep,NaN
4,2018,House,Alabama,District 5,Peter Joffrion,Mo Brooks,101388.0,159063.0,n,y,rep,NaN


In [231]:
rieke.tail()

,cycle,race,state,seat,candidate_name_DEM,candidate_name_REP,dem_votes,rep_votes,dem_incumbent,rep_incumbent,prev_seat,pending
501,2022,Governor,Tennessee,Governor,Martin,Lee,NaN,NaN,n,y,NaN,NaN
502,2022,Governor,Alabama,Governor,Flowers,Ivey,NaN,NaN,n,y,NaN,NaN
503,2022,Governor,South Dakota,Governor,Smith,Noem,NaN,NaN,n,y,NaN,NaN
504,2022,Governor,Idaho,Governor,Heidt,Little,NaN,NaN,n,y,NaN,NaN
505,2022,Governor,Wyoming,Governor,Livingston,Gordon,NaN,NaN,n,y,NaN,NaN


In [232]:
rieke['dem_incumbent'] = rieke['dem_incumbent'].replace({'n': False, 'y': True})
rieke['rep_incumbent'] = rieke['rep_incumbent'].replace({'n': False, 'y': True})

In [233]:
rieke = rieke.rename({'dem_incumbent': 'dem_inc', 'rep_incumbent': 'rep_inc', 'seat': 'district',
                     'cycle': 'year'}, axis=1)

In [234]:
rieke = rieke[~rieke['district'].isin(['Class I', 'Class II', 'Class III', 'Governor'])]
rieke['district'] = rieke['district'].str.lstrip('District ').astype(int)
rieke.head(3)

,year,race,state,district,candidate_name_DEM,candidate_name_REP,dem_votes,rep_votes,dem_inc,rep_inc,prev_seat,pending
0,2018,House,Alabama,1,Robert Kennedy Jr.,Bradley Byrne,89226.0,153228.0,False,True,rep,NaN
1,2018,House,Alabama,2,Tabitha Isner,Martha Roby,86931.0,138879.0,False,True,rep,NaN
2,2018,House,Alabama,3,Mallory Hagan,Mike Rogers,83996.0,147770.0,False,True,rep,NaN


In [235]:
data.shape

(1742, 11)

In [236]:
data = pd.merge(left=data, right=rieke[['year', 'state', 'district', 'dem_inc', 'rep_inc']], on=['year', 'state', 'district'],
               how='left')
data.head(3)

,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes,dem_inc,rep_inc
0,2018,Alabama,AL,False,1,89226.0,153228.0,Robert Kennedy Jr,Bradley Byrne,242617,242454.0,False,True
1,2018,Alabama,AL,False,2,86931.0,138879.0,Tabitha Isner,Martha Roby,226230,225810.0,False,True
2,2018,Alabama,AL,False,3,83996.0,147770.0,Mallory Hagan,Mike Rogers,231915,231766.0,False,True


In [237]:
data.shape

(1742, 13)

In [238]:
## For now, fill in True for NaN entries; use Wikipedia and Excel to discern and fill in entries where it's False (for dem_inc, rep_inc)
data['dem_inc'] = data['dem_inc'].fillna(True)
data['rep_inc'] = data['rep_inc'].fillna(True)
data['dem_inc'].isna().any()

np.False_

In [239]:
data['dem_pct_2p'] = 100 * data['dem'] / data['2party_votes']
data['rep_pct_2p'] = 100 * data['rep'] / data['2party_votes']
data.head(3)

,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes,dem_inc,rep_inc,dem_pct_2p,rep_pct_2p
0,2018,Alabama,AL,False,1,89226.0,153228.0,Robert Kennedy Jr,Bradley Byrne,242617,242454.0,False,True,36.801208,63.198792
1,2018,Alabama,AL,False,2,86931.0,138879.0,Tabitha Isner,Martha Roby,226230,225810.0,False,True,38.497409,61.502591
2,2018,Alabama,AL,False,3,83996.0,147770.0,Mallory Hagan,Mike Rogers,231915,231766.0,False,True,36.241727,63.758273
